# Raw Data Preparation: Satellite Bands and CAMS Integration

## Overview
This document outlines the preprocessing pipeline for the solar forecasting dataset. The workflow ingests combined 15-channel satellite imagery and CAMS (Copernicus Atmosphere Monitoring Service) data, segments the observations by diurnal cycles, and temporally aligns the datasets using fuzzy matching.

## 1. Initial Data Ingestion
The pipeline begins by loading the combined 15-channel satellite band data to verify its structure and temporal bounds.
* **Source File:** `Data/bands/combined_15ch_full.npz`
* **Data Dimensions:** `(154779, 15, 50, 50)`
* **End Date:** `2023-12-31 23:50:00`

## 2. Diurnal Segmentation
To account for differing solar and atmospheric conditions, the combined dataset is explicitly split into day and night/twilight subsets. Any timesteps that result entirely in `NaN` values during this split are dropped to maintain data integrity.
* **Daytime Output:** `combined_15ch_full_day_no_twillight.npz` *(Strict daylight, no twilight)*
* **Nighttime Output:** `combined_15ch_full_night_twillight.npz` *(Nighttime and twilight hours)*

## 3. Temporal Alignment
The segmented satellite bands are then paired with CAMS irradiance data (`ghi_grid2500_2ch_15mins.npz`, comprising 140,214 initial timestamps). Because sensor readings do not always perfectly align, a fuzzy matching algorithm is applied with a maximum tolerance of 6 minutes. One-to-many matching is permitted.

### Daytime Dataset Pairing
* **Input Source:** `combined_15ch_full_day_no_twillight.npz`
* **Matched Samples:** 67,638 successful pairs
* **Maximum Time Delta (|Δt|):** 5 minutes
* **Bands Output:** `Data/bands/modeldata/bands_day_paired_30mins_new_fuzzy_no_twillight.npz`
* **CAMS Output:** `Data/CAMS/modeldata/cams_day_paired_30mins_new_fuzzy_no_twillight.npz`

### Nighttime and Twilight Dataset Pairing
* **Input Source:** `combined_15ch_full_night_twillight.npz`
* **Matched Samples:** 87,016 successful pairs
* **Maximum Time Delta (|Δt|):** 5 minutes
* **Bands Output:** `Data/bands/modeldata/bands_night_paired_30mins_new_fuzzy_twillight.npz`
* **CAMS Output:** `Data/CAMS/modeldata/cams_night_paired_30mins_new_fuzzy_twillight.npz`

In [2]:
import numpy as np
%load_ext autoreload
%autoreload 2
from download.preproceessing import build_combined_npz,split_day_night_npz,pair_and_save_cams_bands

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Initial Data Ingestion
The pipeline begins by loading the combined 15-channel satellite band data to verify its structure and temporal bounds.
* **Source File:** `Data/bands/combined_15ch_full.npz`
* **Data Dimensions:** `(154779, 15, 50, 50)`
* **End Date:** `2023-12-31 23:50:00`

In [ ]:
build_combined_npz(root="../Data/2kmhk", out="combined_15ch_full.npz")

In [5]:
z = np.load("../Data/bands/combined_15ch_full.npz")
z.keys()

KeysView(NpzFile '../Data/bands/combined_15ch_full.npz' with keys: data, time_min)

## 2. Diurnal Segmentation
To account for differing solar and atmospheric conditions, the combined dataset is explicitly split into day and night/twilight subsets. Any timesteps that result entirely in `NaN` values during this split are dropped to maintain data integrity.
* **Daytime Output:** `combined_15ch_full_day_no_twillight.npz` *(Strict daylight, no twilight)*
* **Nighttime Output:** `combined_15ch_full_night_twillight.npz` *(Nighttime and twilight hours)*

In [ ]:
split_day_night_npz(
    "../Data/bands/combined_15ch_full.npz",
    "../Data/bands/combined_15ch_full_day_no_twillight.npz",
    "../Data/bands/combined_15ch_full_night_twillight.npz",
    day_thresh = 80.0,
    night_thresh = 90.0,
    twilight="night",          # strict day/night only
    drop_nan_timesteps=True,  # remove timesteps that become all-NaN 
)

In [ ]:
day_bands = np.load("../Data/bands/combined_15ch_full_day_no_twillight.npz")
day_data = day_bands["data"]
day_time = day_bands["time_min"]

## 3. Temporal Alignment
The segmented satellite bands are then paired with CAMS irradiance data (`ghi_grid2500_2ch_15mins.npz`, comprising 140,214 initial timestamps). Because sensor readings do not always perfectly align, a fuzzy matching algorithm is applied with a maximum tolerance of 6 minutes.


### Daytime Dataset Pairing

In [ ]:
# load CAMS data
cams = np.load("../Data/CAMS/ghi_grid2500_2ch_15mins.npz")
cams_data = cams["data"]
cams_time = cams["time"]

# pair CAMS and bands data for daytime only
pair_and_save_cams_bands(
    bands_npz_path = "../Data/bands/combined_15ch_full_day_no_twillight.npz",
    cams_npz_path = "../Data/CAMS/ghi_grid2500_2ch_15mins.npz",
    out_cams_npz_path="../Data/CAMS/modeldata/cams_day_paired_no_twillight.npz",
    out_bands_npz_path= "../Data/bands/modeldata/bands_day_paired_no_twillight.npz",
    tol_min=6,
    filter_cams = False, 
    one_to_one=False 
)

Random 20 matched pairs:
00  bands=2023-09-18T03:30  cams=2023-09-18T03:30  |Δ|=0 min
01  bands=2021-11-29T07:30  cams=2021-11-29T07:30  |Δ|=0 min
02  bands=2021-10-15T00:20  cams=2021-10-15T00:15  |Δ|=5 min
03  bands=2023-10-13T00:50  cams=2023-10-13T00:45  |Δ|=5 min
04  bands=2022-07-10T06:40  cams=2022-07-10T06:45  |Δ|=5 min
05  bands=2022-10-18T07:10  cams=2022-10-18T07:15  |Δ|=5 min
06  bands=2022-08-28T08:10  cams=2022-08-28T08:15  |Δ|=5 min
07  bands=2023-11-24T05:20  cams=2023-11-24T05:15  |Δ|=5 min
08  bands=2022-11-24T00:50  cams=2022-11-24T00:45  |Δ|=5 min
09  bands=2022-12-10T01:50  cams=2022-12-10T01:45  |Δ|=5 min
10  bands=2021-07-08T08:00  cams=2021-07-08T08:00  |Δ|=0 min
11  bands=2023-06-10T05:00  cams=2023-06-10T05:00  |Δ|=0 min
12  bands=2022-11-18T04:10  cams=2022-11-18T04:15  |Δ|=5 min
13  bands=2021-01-22T07:30  cams=2021-01-22T07:30  |Δ|=0 min
14  bands=2022-08-12T01:20  cams=2022-08-12T01:15  |Δ|=5 min
15  bands=2021-03-30T23:40  cams=2021-03-30T23:45  |Δ|=5 min

{'n_matched': 67638, 'max_abs_dt_min': 5}

### Nighttime and Twilight Dataset Pairing

In [ ]:
# pair CAMS and bands data for nighttime only
pair_and_save_cams_bands(
    bands_npz_path = "../Data/bands/combined_15ch_full_night_twillight.npz",
    cams_npz_path = "../Data/CAMS/ghi_grid2500_2ch_15mins.npz",
    out_cams_npz_path="../Data/CAMS/modeldata/cams_night_paired_twillight.npz",
    out_bands_npz_path= "Data/bands/modeldata/bands_night_paired_twillight.npz",
    tol_min=6,
    filter_cams = False, 
    one_to_one=True
)

Random 20 matched pairs:
00  bands=2023-09-03T22:10  cams=2023-09-03T22:15  |Δ|=5 min
01  bands=2021-08-25T16:00  cams=2021-08-25T16:00  |Δ|=0 min
02  bands=2021-07-26T13:30  cams=2021-07-26T13:30  |Δ|=0 min
03  bands=2023-10-06T12:40  cams=2023-10-06T12:45  |Δ|=5 min
04  bands=2022-01-25T19:10  cams=2022-01-25T19:15  |Δ|=5 min
05  bands=2022-06-15T10:20  cams=2022-06-15T10:15  |Δ|=5 min
06  bands=2022-04-03T21:10  cams=2022-04-03T21:15  |Δ|=5 min
07  bands=2023-11-23T20:10  cams=2023-11-23T20:15  |Δ|=5 min
08  bands=2022-08-02T19:40  cams=2022-08-02T19:45  |Δ|=5 min
09  bands=2022-08-21T14:10  cams=2022-08-21T14:15  |Δ|=5 min
10  bands=2023-04-14T13:00  cams=2023-04-14T13:00  |Δ|=0 min
11  bands=2023-04-03T10:40  cams=2023-04-03T10:45  |Δ|=5 min
12  bands=2022-07-26T19:40  cams=2022-07-26T19:45  |Δ|=5 min
13  bands=2021-01-12T16:00  cams=2021-01-12T16:00  |Δ|=0 min
14  bands=2021-12-03T12:20  cams=2021-12-03T12:15  |Δ|=5 min
15  bands=2023-09-18T21:00  cams=2023-09-18T21:00  |Δ|=0 min

{'n_matched': 59236, 'max_abs_dt_min': 5}